<a href="https://colab.research.google.com/github/Kokila-16/1.Python/blob/main/Trans_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, MarianTokenizer
from datasets import load_dataset

In [2]:
model_checkpoint = "Helsinki-NLP/opus-mt-en-mul"
tokenizer = MarianTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/707k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [3]:
raw_datasets = load_dataset("Hemanth-thunder/en_ta")
print(raw_datasets)

README.md:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

corpus.bcn.train 166k.csv: reconstructing file:   0%|          |  0.00B / 99.2MB            

corpus.bcn.train 166k.csv: downloading bytes:           |  0.00B            

en-ta train 118k.csv: reconstructing file:   0%|          |  0.00B / 75.8MB            

en-ta train 118k.csv: downloading bytes:           |  0.00B            

...

corpus.bcn.test%202k.csv:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/285630 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'en', 'ta'],
        num_rows: 285630
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'en', 'ta'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['Unnamed: 0', 'en', 'ta'],
        num_rows: 2000
    })
})


In [4]:
max_input_length = 128
max_target_length = 128
learning_rate=2e-5
batch_size=2
epochs=2
source_lang = "en"
target_lang = "ta"

In [5]:
def preprocess_function(examples):
    inputs = [ex for ex in examples[source_lang]]
    targets = [ex for ex in examples[target_lang]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")

    # Note: No need to use tokenizer.as_target_tokenizer() in PyTorch. This was a TensorFlow specific step.
    labels = tokenizer(targets, max_length=max_target_length, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [7]:
from google.colab import files
uploaded = files.upload()

Saving engtamilTrain.csv to engtamilTrain.csv


In [8]:
import pandas as pd
from datasets import Dataset, DatasetDict

df = pd.read_csv("engtamilTrain.csv")
raw_datasets = DatasetDict({"train": Dataset.from_pandas(df)})
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'en', 'ta'],
        num_rows: 166871
    })
})


In [9]:
from transformers import MarianTokenizer

tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-mul")

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [10]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/166871 [00:00<?, ? examples/s]

In [11]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["en", "ta"])
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/166871 [00:00<?, ? examples/s]

In [12]:
from transformers import AutoModelForSeq2SeqLM, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments

In [13]:
model_checkpoint = "Helsinki-NLP/opus-mt-en-mul"
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  310MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  310MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [15]:
tokenized_datasets = tokenized_datasets["train"].train_test_split(test_size=0.1)

In [16]:
from datasets import DatasetDict

tokenized_datasets = DatasetDict({
    "train": tokenized_datasets["train"],
    "validation": tokenized_datasets["test"]
})

In [17]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=False,
)
# Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]